# Ciringoni M4 E7 ekspor checkpoint historis

Jalankan seleksi validation, kalibrasi, ekspor OpenVINO, serta benchmark di Kaggle. Bobot pasangan native berasal dari pelatihan lokal 30 epoch pada 18 November 2026.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = "https://github.com/Jordyee/ciringoni-data-mining.git"
SOURCE = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
PROJECT = Path("/tmp/ciringoni-source/Data_Mining_Ciringoni_M1")
SCRIPTS = PROJECT / "hasil_m2"

def execute(*args):
    subprocess.run([str(arg) for arg in args], cwd=PROJECT, check=True)

def input_file(name):
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f"Expected one mounted {name}, found {len(paths)}"
    return paths[0]

subprocess.run(["git", "clone", "--quiet", REPO, str(Path("/tmp/ciringoni-source"))], check=True)
execute("git", "checkout", "--quiet", SOURCE)


## Lingkungan dan input


In [ ]:
execute(sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.210", "pycocotools==2.0.11", "openvino==2026.4.1", "nncf==3.4.0")
print("Pinned OpenVINO and COCO dependencies installed.", flush=True)
from tarfile import open as tar_open
from zipfile import ZipFile

NATIVE = Path("/tmp/ciringoni-native")
NATIVE.mkdir()
archive_paths = list(INPUT.rglob("native_train_val.tar")) + list(INPUT.rglob("native_train_val.zip"))
if archive_paths:
    assert len(archive_paths) == 1
    archive = archive_paths[0]
    if archive.suffix == ".zip":
        with ZipFile(archive) as packed:
            packed.extractall(NATIVE)
    else:
        with tar_open(archive) as packed:
            packed.extractall(NATIVE, filter="data")
    if not (NATIVE / "640").is_dir():
        roots = [path.parent for path in NATIVE.rglob("640") if (path / "images").is_dir()]
        assert len(roots) == 1
        NATIVE = roots[0]
else:
    from concurrent.futures import ThreadPoolExecutor

    source_dataset = input_file("audit_pasangan_e2.json").parent
    roots = [root for root in (source_dataset, source_dataset / "native_train_val")
             if (root / "640/images/val").is_dir() and (root / "1280/images/val").is_dir()]
    assert len(roots) == 1, f"Expected the mounted native input: {roots}"
    copies = []
    for resolution in ("640", "1280"):
        source = roots[0] / resolution
        target = NATIVE / resolution
        (target / "images/val").mkdir(parents=True)
        (target / "images/train").symlink_to(source / "images/train", target_is_directory=True)
        (target / "labels").mkdir()
        for split in ("train", "val"):
            (target / "labels" / split).symlink_to(source / "labels" / split, target_is_directory=True)
        images = sorted((source / "images/val").glob("*.jpg"))
        assert len(images) == 356
        copies.extend((image, target / "images/val" / image.name) for image in images)
    print("Staging 712 validation images. Train images remain mounted for calibration.", flush=True)
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(lambda paths: shutil.copyfile(*paths), copies))
    print("Both validation sets staged.", flush=True)
if not (NATIVE / "audit_pasangan_e2.json").is_file():
    shutil.copyfile(input_file("audit_pasangan_e2.json"), NATIVE / "audit_pasangan_e2.json")
for size in (640, 1280):
    root = NATIVE / str(size)
    assert root.is_dir() and not (root / "images/test").exists()
    (root / "data.yaml").write_text(
        json.dumps({"path": str(root), "train": "images/train", "val": "images/val", "names": ["infected"]})
    )
for target, name in [(PROJECT / "data", "yolo26n.pt"), (PROJECT / "hasil_m1", "coco_val_gt.json")]:
    target.mkdir(exist_ok=True)
    source = NATIVE / name
    shutil.copyfile(source if source.is_file() else input_file(name), target / name)
with (PROJECT / "data/yolo26n.pt").open("rb") as stream:
    assert hashlib.file_digest(stream, "sha256").hexdigest() == "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"
from tarfile import open as tar_open
bundles = list(INPUT.rglob("reference-checkpoints.tar"))
if bundles:
    assert len(bundles) == 1
    with tar_open(bundles[0]) as packed:
        packed.extractall(Path("/tmp/ciringoni-reference"), filter="data")

def reference_manifest(name):
    files = list(INPUT.rglob(name)) + list(Path("/tmp/ciringoni-reference").rglob(name))
    assert len(files) == 1, f"Expected one {name} in reference artifacts"
    return files[0]


## Seleksi resolusi dengan validation

Ambang promosi 1280 terhadap 640 adalah kenaikan AP-small minimal 0,005, penurunan mAP50–95 maksimal 0,005, dan penurunan recall maksimal 0,02.


In [ ]:
import csv
import pandas as pd

OUTPUT = WORK / "m4"
OUTPUT.mkdir()
train_manifest_path = reference_manifest("m4-training-reference.json")
training = json.loads(train_manifest_path.read_text())
assert not training["trained_on_kaggle"]
shutil.copyfile(train_manifest_path, OUTPUT / train_manifest_path.name)
assert training["epochs"] == 30 and training["batch"] == 4 and training["seed"] == 42
checkpoints = {}
scores = {}
rows = []
for size in (640, 1280):
    info = training["runs"][str(size)]
    path = train_manifest_path.parent / info["path"]
    with path.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == info["best_pt_sha256"]
    assert len((path.parents[1] / "results.csv").read_text().splitlines()) == 31
    checkpoints[size] = path
    result = OUTPUT / f"native_{size}"
    execute(sys.executable, SCRIPTS / "evaluasi_final.py", "val", path, NATIVE, result, "--size", size)
    scores[size] = json.loads((result / "metrics.json").read_text())
    assert scores[size]["dataset"]["images"] == 356 and scores[size]["dataset"]["boxes"] == 2205
    rows.append({"id": f"native_{size}_30ep", "split": "val", **scores[size]})
assert scores[640]["dataset"] == scores[1280]["dataset"]
assert scores[640]["runtime"] == scores[1280]["runtime"]
deltas = {key: scores[1280][key] - scores[640][key] for key in ("ap_small", "ap_50_95", "recall")}
SIZE = 1280 if deltas["ap_small"] >= 0.005 and deltas["ap_50_95"] >= -0.005 and deltas["recall"] >= -0.02 else 640
print("Native resolution selected by validation:", SIZE, deltas)


## Kalibrasi INT8 dan perbandingan OpenVINO

Kalibrasi memakai 320 gambar train dengan seed 42. Pilih INT8 jika penurunan AP-small dan mAP50–95 masing-masing maksimal 0,005, penurunan recall maksimal 0,02, dan p95 CPU membaik.


In [ ]:
EXPORT = OUTPUT / "export"
execute(sys.executable, SCRIPTS / "ekspor_final.py", checkpoints[SIZE], NATIVE, EXPORT, "--size", SIZE)
comparison = json.loads((EXPORT / "comparison.json").read_text())
variants = comparison["exports"]
fp32, int8 = variants["fp32"], variants["int8"]
deltas_int8 = {key: int8[key] - fp32[key] for key in ("ap_small", "ap_50_95", "recall", "cpu_p95_ms")}
PRECISION = "int8" if (deltas_int8["ap_small"] >= -0.005 and deltas_int8["ap_50_95"] >= -0.005
                       and deltas_int8["recall"] >= -0.02 and deltas_int8["cpu_p95_ms"] < 0) else "fp32"
for precision, result in variants.items():
    rows.append({"id": f"openvino_{precision}", "split": "val", **result})
keys = ["id", "split", "size", "ap_small", "ap_50_95", "ap_50", "precision", "recall",
        "cpu_median_ms", "cpu_p95_ms", "model_bytes", "fake_quantize_ops"]
with (OUTPUT / "eksperimen.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
decision = {"training_performed_on_kaggle": False, "checkpoint_origin": "Historical 2026-11-18 local training",
            "size": SIZE, "precision": PRECISION, "resolution_deltas_1280_minus_640": deltas,
            "export_deltas_int8_minus_fp32": deltas_int8}
(OUTPUT / "selection.json").write_text(json.dumps(decision, indent=2) + "\n")
display(pd.read_csv(OUTPUT / "eksperimen.csv"))
print("Selection:", json.dumps(decision, indent=2))


## Kunci sebelum test

Tahap ini menggunakan fingerprint test dari audit yang sudah diverifikasi. Dataset test tidak dipasang ke notebook ini. Commit lock hasil run sebelum membuat versi notebook test.


In [ ]:
from datetime import datetime, timedelta, timezone
sys.path.insert(0, str(SCRIPTS))
from evaluasi_final import lock_config

# Reuse the fingerprint of the previously audited held-out split. No test data is mounted or inferred here.
test_fingerprint = json.loads((PROJECT.parent / "test-lock.json").read_text())["test"]
row = variants[PRECISION]
models = []
for candidate in EXPORT.rglob("*.xml"):
    with candidate.open("rb") as stream:
        if row["model_sha256"].get(candidate.name) == hashlib.file_digest(stream, "sha256").hexdigest():
            models.append(candidate.parent)
assert len(models) == 1
lock = lock_config(models[0], EXPORT / "candidate.pt", (test_fingerprint,),
                   EXPORT / PRECISION / "metrics.json", SIZE, PRECISION)
lock["locked_at_wib"] = datetime.now(timezone(timedelta(hours=7))).isoformat(timespec="seconds")
LOCK = OUTPUT / "test-lock-kaggle.json"
LOCK.write_text(json.dumps(lock, indent=2) + "\n", encoding="utf-8")
with LOCK.open("rb") as stream:
    print("Commit this lock before running held-out inference.", hashlib.file_digest(stream, "sha256").hexdigest())
